In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import os.path as osp

from pathlib import Path
from pprint import pprint

import sqlparse
import sys

root = Path.cwd().parent 
if str(root) not in sys.path:    
    sys.path.append(str(root))

In [3]:
import duckdb 

from src import logger
from src.config import CFG
from src.db.election_db import ElectionDB
from src.agents.agent import VIOLATIONS_COUNTER, QueryIntent, db_client
from src.agents.hybrid_agent import HybridAgent
from langchain_core.messages import HumanMessage

/Users/dric/projects/research/LLMs/chat_app/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [4]:
from prometheus_client import start_http_server, Counter, REGISTRY

nb_port = 8002
try:
    start_http_server(nb_port)
    print(f"Notebook metrics live on port {nb_port}")
except OSError:
    print(f"Port {nb_port} is already in use")


Notebook metrics live on port 8002


In [5]:
agent = HybridAgent()

agent()

2026-03-27 15:14:15,098 [INFO] LocalLLMStack: Agent Initialized


In [6]:
agent.client.streaming, agent.client.openai_api_base, agent.client.model_name

(False, 'http://127.0.0.1:8000/v1', 'Qwen/Qwen3-4B-Instruct-2507')

### Inspecting tools

In [7]:
for idx, t in enumerate(agent.tools):
    print(f" {idx+1}) {t.name}: \t{t.description[:30]}...")

 1) describe_table: 	Use this to get column names, ...
 2) execute_read_query: 	Executes a SQL SELECT query an...
 3) get_db_schema: 	Returns the schema (table name...
 4) list_tables: 	Get a list of all available ta...
 5) sample_data: 	Fetch the first 3 rows of a ta...
 6) validate_sql: 	Validating SQL syntax and logi...
 7) search_database: 	Performs hybrid (Full-Text and...


## Check DB search operations

In [8]:
agent.sql_expert.list_tables.args

{'reasoning': {'description': 'Explanation of why this tool is being called and what information is sought.',
  'title': 'Reasoning',
  'type': 'string'}}

In [9]:
agent.sql_expert.list_tables.invoke({
    "reasoning": "just listing tables"
})

2026-03-27 14:13:38,438 [INFO] LocalLLMStack: LLM Reasoning (list_tables): just listing tables


'{"name":{"0":"candidate","1":"constituency","2":"embeddings","3":"entity_alias","4":"party","5":"region","6":"result","7":"turnout","8":"vw_party","9":"vw_rag_descriptions","10":"vw_results","11":"vw_turnout","12":"vw_winners"}}'

In [10]:
agent.sql_expert.sample_data.args

{'reasoning': {'description': 'Explanation of why this tool is being called and what information is sought.',
  'title': 'Reasoning',
  'type': 'string'},
 'table_name': {'description': 'The exact name of the table to inspect. Mandatory.',
  'title': 'Table Name',
  'type': 'string'}}

In [11]:
agent.sql_expert.sample_data.invoke({
    "reasoning": "sampling some data to see the values",
    "table_name": "vw_results",
    "num_samples": 10
})

2026-03-27 14:13:38,553 [INFO] LocalLLMStack: LLM Reasoning (sample_data [vw_results]): sampling some data to see the values


,RESULT_ID,REGION_NAME,CONSTITUENCY_NUM,CONSTITUENCY_TITLE,CANDIDATE_NAME,PARTY_NAME,NUM_VOTES,PCT_SCORE,IS_WINNER
0,584,LAME,141,"ADZOPE, COMMUNE",ACHI PATRICK JEROME,RHDP,7246,82.75,True
1,560,KABADOUGOU,120,"FENGOLO, MADINANI ET N'GOLOBLASSO, COMMUNES ET...",FOFANA LASSANA,INDEPENDANT,4430,38.12,False
2,614,LAME,146,"ABOISSO-COMOE, ALEPE, ALLOSSO, DANGUIRA ETOGHL...",YAPO MARTIAL AUGUSTE,INDEPENDANT,889,6.51,False
3,31,AGNEBY-TIASSA,005,"GOMON ET SIKENSI, COMMUNES ET SOUS-PREFECTURES",ADOU JOSE ANTONIO,INDEPENDANT,64,0.61,False
4,855,SAN-PEDRO,177,"DJOUROUTOU ET GRABO, COMMUNES ET SOUS-PREFECTURES",GNEPA DOUBOYOU ALASSANE,INDEPENDANT,367,8.41,False


In [12]:
agent.sql_expert.describe_table.args

{'reasoning': {'description': 'Explanation of why this tool is being called and what information is sought.',
  'title': 'Reasoning',
  'type': 'string'},
 'table_name': {'description': 'The exact name of the table to inspect. Mandatory.',
  'title': 'Table Name',
  'type': 'string'}}

In [13]:
agent.sql_expert.describe_table.invoke({
    "reasoning": "describing the table structure",
    "table_name": "vw_party"
})

2026-03-27 14:13:38,687 [INFO] LocalLLMStack: LLM Reasoning (describe_table [vw_party]): describing the table structure


'   column_name column_type null   key default extra\n0   PARTY_NAME     VARCHAR  YES  None    None  None\n1  REGION_NAME     VARCHAR  YES  None    None  None\n2  TOTAL_VOTES     HUGEINT  YES  None    None  None\n3    SEATS_WON      BIGINT  YES  None    None  None'

### Vector search (VS)

In [14]:
results = db_client.vector_search(
    reasoning="performing some vector search",
    query="Who was elected in the Agneby-Tiassa region from RHDP?", 
    top_k=5
)

for text, chunk_id, score in results:
    print(f"chunk_id {chunk_id}: \t[{score:.2f}] {text}")

2026-03-27 14:13:38,774 [INFO] LocalLLMStack: LLM Reasoning (vector_search): performing some vector search
2026-03-27 14:13:38,883 [INFO] sentence_transformers.SentenceTransformer: Use pytorch device_name: mps
2026-03-27 14:13:38,884 [INFO] sentence_transformers.SentenceTransformer: Load pretrained SentenceTransformer: google/embeddinggemma-300m


Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

2026-03-27 14:13:49,235 [INFO] sentence_transformers.SentenceTransformer: 14 prompts are loaded, with the keys: ['query', 'document', 'BitextMining', 'Clustering', 'Classification', 'InstructionRetrieval', 'MultilabelClassification', 'PairClassification', 'Reranking', 'Retrieval', 'Retrieval-query', 'Retrieval-document', 'STS', 'Summarization']


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

chunk_id 1517: 	[0.76] In the region of AGNEBY-TIASSA, the winningest party was RHDP with 5 seats.
chunk_id 1355: 	[0.70] The party RHDP won a total of 5 seats across AGNEBY-TIASSA. They received 25827 total votes in this region.
chunk_id 1422: 	[0.64] The party FPI won a total of 0 seats across AGNEBY-TIASSA. They received 117 total votes in this region.
chunk_id 1481: 	[0.64] The party ADCI won a total of 0 seats across AGNEBY-TIASSA. They received 2475 total votes in this region.
chunk_id 1460: 	[0.63] The party PDCI-RDA won a total of 0 seats across AGNEBY-TIASSA. They received 5314 total votes in this region.


### Full text search (FTS)

In [15]:
results = db_client.full_text_search(
    query="What was the turnout in tiapoum?", 
    top_k=5
)

for text, chunk_id, score in results:
    print(f"chunk_id {chunk_id}: \t[{score:.2f}] {text}")

chunk_id 589: 	[2.32] Candidate (or candidate group) SANGARE ISSA representing INDEPENDANT ran in NOE, NOUAMOU ET TIAPOUM, COMMUNES ET SOUS-PREFECTURES. They received 3360 votes, finishing in rank 1. Outcome: ELECTED.
chunk_id 592: 	[2.32] Candidate (or candidate group) KONATE SEYDOU representing INDEPENDANT ran in NOE, NOUAMOU ET TIAPOUM, COMMUNES ET SOUS-PREFECTURES. They received 522 votes, finishing in rank 4. Outcome: NOT ELECTED.
chunk_id 590: 	[2.32] Candidate (or candidate group) BAKARY KAMARA representing RHDP ran in NOE, NOUAMOU ET TIAPOUM, COMMUNES ET SOUS-PREFECTURES. They received 2933 votes, finishing in rank 2. Outcome: NOT ELECTED.
chunk_id 591: 	[2.27] Candidate (or candidate group) ANDI ADOUALOU ROLAND representing INDEPENDANT ran in NOE, NOUAMOU ET TIAPOUM, COMMUNES ET SOUS-PREFECTURES. They received 740 votes, finishing in rank 3. Outcome: NOT ELECTED.
chunk_id 593: 	[2.27] Candidate (or candidate group) DONGO ABDOUL FATAOU representing INDEPENDANT ran in NOE, NOUAM

### Hybrid search

In [16]:
results = db_client.hybrid_search(
    query="Who was elected in Tiapoum region?", 
    top_k=5
)

results

2026-03-27 14:13:50,147 [INFO] sentence_transformers.SentenceTransformer: Use pytorch device_name: mps
2026-03-27 14:13:50,148 [INFO] sentence_transformers.SentenceTransformer: Load pretrained SentenceTransformer: google/embeddinggemma-300m


Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

2026-03-27 14:13:58,724 [INFO] sentence_transformers.SentenceTransformer: 14 prompts are loaded, with the keys: ['query', 'document', 'BitextMining', 'Clustering', 'Classification', 'InstructionRetrieval', 'MultilabelClassification', 'PairClassification', 'Reranking', 'Retrieval', 'Retrieval-query', 'Retrieval-document', 'STS', 'Summarization']


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,TEXT_CHUNK,CHUNK_ID,weighted_rrf_score
0,"In the region of WORODOUGOU, the winningest pa...",1526,0.015111
1,"In the region of BAGOUE, the winningest party ...",1522,0.014995
2,"In the region of KABADOUGOU, the winningest pa...",1520,0.014994
3,"In the region of AGNEBY-TIASSA, the winningest...",1517,0.014667
4,"In the region of DISTRICTAUTONOMED'ABIDJAN, th...",1507,0.014198


## Check Agent-specific operations

### Testing Security Sandbox

In [17]:
sql = "SELECT * FROM REGION"
parsed = sqlparse.parse(sql)

clean_sql = parsed[0]

clean_sql, clean_sql.value

(<Statement 'SELECT...' at 0x12A336C10>, 'SELECT * FROM REGION')

In [18]:
agent.sql_expert.validate_sql.args

{'sql': {'title': 'Sql', 'type': 'string'},
 'forbidden': {'items': {}, 'title': 'Forbidden', 'type': 'array'},
 'reasoning': {'title': 'Reasoning', 'type': 'string'},
 'config': {'default': None, 'title': 'Config'}}

In [19]:
try:
    agent.forbidden = ["SELECT"] # Attempt to change the list
    agent._forbidden = [] # Attempt to change the list

except AttributeError as e:
    print(f"✅ Success: Modification blocked - {e}")

# Manual validation test
for kw in ["DROP", "ALTER", "UPDATE"]:
    malicious = f"SELECT * FROM results; {kw} TABLE turnout;"
    is_safe, out_message = agent.sql_expert.validate_sql.invoke({
        "reasoning": "validating SQL query",
        "sql": malicious,
        "forbidden": agent.sql_expert.forbidden
    })
    print(f"Is {kw} query safe? {is_safe} (Expected: False)")
    print(f"Message: {out_message}")
    print()

for kw in ["DROP", "ALTER", "UPDATE"]:
    malicious = f"{kw} TABLE result;"
    is_safe, out_message = agent.sql_expert.validate_sql.invoke({
        "reasoning": "validating SQL query",
        "sql": malicious,
        "forbidden": agent.sql_expert.forbidden
    })
    print(f"Is {kw} query safe?\t {is_safe} (Expected: False)")
    print(f"Message: {out_message}")
    print()

✅ Success: Modification blocked - The 'forbidden' attribute is sealed and cannot be modified.
2026-03-27 14:13:59,163 [INFO] LocalLLMStack: LLM Reasoning (validate_sql): validating SQL query
Is DROP query safe? False (Expected: False)
Message: Multiple queries detected.

2026-03-27 14:13:59,168 [INFO] LocalLLMStack: LLM Reasoning (validate_sql): validating SQL query
Is ALTER query safe? False (Expected: False)
Message: Multiple queries detected.

2026-03-27 14:13:59,171 [INFO] LocalLLMStack: LLM Reasoning (validate_sql): validating SQL query
Is UPDATE query safe? False (Expected: False)
Message: Multiple queries detected.

2026-03-27 14:13:59,177 [INFO] LocalLLMStack: LLM Reasoning (validate_sql): validating SQL query
Is DROP query safe?	 False (Expected: False)
Message: Forbidden command type 'DROP'.

2026-03-27 14:13:59,183 [INFO] LocalLLMStack: LLM Reasoning (validate_sql): validating SQL query
Is ALTER query safe?	 False (Expected: False)
Message: Forbidden command type 'ALTER'.

2

In [20]:
is_safe, out_message = agent.sql_expert.validate_sql.invoke({
    "reasoning": "validating SQL query",
    "sql": clean_sql.value,
    "forbidden": agent.sql_expert.forbidden
})
print(f"Is query safe? {is_safe} (Expected: True)")
print(f"Message: {out_message}")
print()

2026-03-27 14:13:59,247 [INFO] LocalLLMStack: LLM Reasoning (validate_sql): validating SQL query
Is query safe? False (Expected: True)
Message: Unauthorized table access detected.
SQL: SELECT * FROM REGION



In [21]:
is_safe, out_message = agent.sql_expert.validate_sql.invoke({
    "sql": "SELECT PARTY_NAME, SUM(SCORES) AS total_votes FROM result GROUP BY PARTY_ID ORDER BY total_votes DESC LIMIT 20",
    "reasoning": "validating SQL query",
    "forbidden": agent.sql_expert.forbidden
})

print(f"Is query safe? {is_safe} (Expected: False)")
print(f"Message: {out_message}")

2026-03-27 14:13:59,301 [INFO] LocalLLMStack: LLM Reasoning (validate_sql): validating SQL query
Is query safe? False (Expected: False)
Message: Unauthorized table access detected.
SQL: SELECT PARTY_NAME, SUM(SCORES) AS TOTAL_VOTES FROM RESULT GROUP BY PARTY_ID ORDER BY TOTAL_VOTES DESC LIMIT 20


In [22]:
metric = REGISTRY.get_sample_value('rag_sql_security_violations_total')
print(f"Current Notebook Violations: {metric}")

Current Notebook Violations: 8.0


### Check read query tool

In [23]:
agent.sql_expert.execute_read_query.args

{'sql_query': {'title': 'Sql Query', 'type': 'string'},
 'reasoning': {'title': 'Reasoning', 'type': 'string'},
 'config': {'default': None, 'title': 'Config'}}

In [24]:
sql = "SELECT PARTY_NAME, SUM(SCORES) AS total_votes FROM result GROUP BY PARTY_ID ORDER BY total_votes DESC LIMIT 20"

for out in agent.sql_expert.execute_read_query.invoke({
    "sql_query": sql,
    "reasoning": "checking query"
}): 
    
    if out["type"] == "error":
        prompt= f"""
        Based on this message: {out["content"]}, what can you do to fix this error? \n
        Answer directly with the fixed query keeping the original parameters (e.g. order or limit) unchanged.
        Original query: {sql}
        """

        fixed =  agent.client.invoke([HumanMessage(content=prompt)]).content
        print(fixed)

2026-03-27 14:13:59,445 [INFO] LocalLLMStack: LLM Reasoning (execute_read_query): checking query
2026-03-27 14:13:59,458 [ERROR] LocalLLMStack: Could not execute query.
Binder Error: Referenced column "SCORES" not found in FROM clause!
Candidate bindings: "NUM_VOTES", "CONSTITUENCY_ID", "CONSTITUENCY_NUM", "CONSTITUENCY_TITLE", "PCT_SCORE"

LINE 1: SELECT PARTY_NAME, SUM(SCORES) AS total_votes FROM result GROUP BY PARTY_ID ORDER...
                               ^
SELECT PARTY_NAME, SUM(PCT_SCORE) AS total_votes FROM result GROUP BY PARTY_ID ORDER BY total_votes DESC LIMIT 20


In [25]:
for out in agent.sql_expert.execute_read_query.invoke({
    "sql_query": fixed,
    "reasoning": "checking query"
}): 
    pass

out
    

2026-03-27 14:14:24,279 [INFO] LocalLLMStack: LLM Reasoning (execute_read_query): checking query
2026-03-27 14:14:24,292 [ERROR] LocalLLMStack: Could not execute query.
Binder Error: column "PARTY_NAME" must appear in the GROUP BY clause or must be part of an aggregate function.
Either add it to the GROUP BY list, or use "ANY_VALUE(PARTY_NAME)" if the exact value of "PARTY_NAME" is not important.

LINE 1: SELECT PARTY_NAME, SUM(PCT_SCORE) AS total_votes FROM result GROUP...
               ^


{'type': 'error',
 'content': 'Could not execute query.\nBinder Error: column "PARTY_NAME" must appear in the GROUP BY clause or must be part of an aggregate function.\nEither add it to the GROUP BY list, or use "ANY_VALUE(PARTY_NAME)" if the exact value of "PARTY_NAME" is not important.\n\nLINE 1: SELECT PARTY_NAME, SUM(PCT_SCORE) AS total_votes FROM result GROUP...\n               ^'}

In [26]:
out["content"]

'Could not execute query.\nBinder Error: column "PARTY_NAME" must appear in the GROUP BY clause or must be part of an aggregate function.\nEither add it to the GROUP BY list, or use "ANY_VALUE(PARTY_NAME)" if the exact value of "PARTY_NAME" is not important.\n\nLINE 1: SELECT PARTY_NAME, SUM(PCT_SCORE) AS total_votes FROM result GROUP...\n               ^'

### Testing intent routing

In [27]:
test_queries = [
    ("Who got the most votes in Abidjan?", "RANKING"),
    ("What is the total turnout?", "AGGREGATION"),
    ("How many colors are there in the rainbow?", "INVALID"),
    ("Tell me about the history of the 2020 election.", "GENERAL"),
    ("What is the weather in Paris?", "INVALID")
]

for query, expected in test_queries:
    intent = agent._get_intent(query)
    print(f"Query: {query} \n> Detected: {intent} (Expected: {expected})\n")
    print()


2026-03-27 14:14:24,379 [INFO] LocalLLMStack: Formatting input to LLM
Query: Who got the most votes in Abidjan? 
> Detected: QueryIntent.RANKING (Expected: RANKING)


2026-03-27 14:14:31,653 [INFO] LocalLLMStack: Formatting input to LLM
Query: What is the total turnout? 
> Detected: QueryIntent.AGGREGATION (Expected: AGGREGATION)


2026-03-27 14:14:35,490 [INFO] LocalLLMStack: Formatting input to LLM
Query: How many colors are there in the rainbow? 
> Detected: QueryIntent.INVALID (Expected: INVALID)


2026-03-27 14:14:38,270 [INFO] LocalLLMStack: Formatting input to LLM
Query: Tell me about the history of the 2020 election. 
> Detected: QueryIntent.GENERAL (Expected: GENERAL)


2026-03-27 14:14:41,710 [INFO] LocalLLMStack: Formatting input to LLM
Query: What is the weather in Paris? 
> Detected: QueryIntent.INVALID (Expected: INVALID)




### SQLAgent 

In [7]:
# import sys

# # Test a raw stream
# for chunk in agent.client.stream("Write a 2-sentence story about PhD."):
#     sys.stdout.write(chunk.content)
#     sys.stdout.flush()


In [8]:
%%time 

query = "What is the total votes by party ?"
intent=QueryIntent.AGGREGATION

if CFG.IS_STREAM:
    print(f"--- Starting Stream for: {query} ---\n")
    
    for chunk in agent.sql_expert.generate_sql(query, intent):
        if chunk["type"] == "status":
            print(f"\n\n[STATUS]: {chunk['content']}", end="", flush=True)
        
        elif chunk["type"] == "token":
            print(chunk["content"], end="", flush=True)
            
        elif chunk["type"] == "action":
            # Visual separator for tool calls
            print(f"\n\n🛠️  [SYSTEM]: {chunk['content']}", end="", flush=True)
            
        elif chunk["type"] == "final_sql":
            print(f"\n\n🚀 [FINAL SQL]:\n{chunk['content']}", end="", flush=True)
            final_sql = chunk['content']
            print(f"\n\n🚀 [SUCCESS]: SQL Captured.")
    
        elif chunk["type"] == "error":
            print(f"\n❌ [ERROR]: {chunk['content']}", end="", flush=True)

else:
    for out in agent.sql_expert.generate_sql(query, intent):
        print(out)
        response = out

2026-03-27 15:14:15,256 [INFO] LocalLLMStack: Attempting SQL generation [Max iterations=18]
{'type': 'status', 'content': 'Attempting SQL generation [Max iterations=18]'}
2026-03-27 15:14:15,257 [INFO] LocalLLMStack: 
[🚶🏼‍➡️ Walking down the SQL path...step 1/18]
{'type': 'status', 'content': '\n[🚶🏼\u200d➡️ Walking down the SQL path...step 1/18]'}
2026-03-27 15:14:52,447 [INFO] LocalLLMStack: Extracting tool calls
2026-03-27 15:14:52,449 [INFO] LocalLLMStack: Using Tool list_tables. Args: {'reasoning': "To identify the relevant tables that might contain information about votes and parties, I need to first list all available tables in the database. This will help me locate tables such as 'vw_results', 'vw_party', or any others that could hold data on votes and party affiliations."}
2026-03-27 15:14:52,452 [INFO] LocalLLMStack: LLM Reasoning (list_tables): To identify the relevant tables that might contain information about votes and parties, I need to first list all available tables in 

In [9]:
response

{'type': 'final_sql',
 'content': 'SELECT \n    PARTY_NAME,\n    SUM(NUM_VOTES) AS TOTAL_VOTES\nFROM \n    vw_results\nGROUP BY \n    PARTY_NAME\nORDER BY \n    TOTAL_VOTES DESC\nLIMIT 20;',
 'sanitized_query': 'SELECT\n    PARTY_NAME,\n    SUM(NUM_VOTES) AS TOTAL_VOTES\nFROM\n    vw_results\nGROUP BY\n    PARTY_NAME\nORDER BY\n    TOTAL_VOTES DESC\nLIMIT 20',
 'steps': ['Using Tool list_tables. Args: {\'reasoning\': "To identify the relevant tables that might contain information about votes and parties, I need to first list all available tables in the database. This will help me locate tables such as \'vw_results\', \'vw_party\', or any others that could hold data on votes and party affiliations."} [Completed in 0.03228s]',
  'Using Tool describe_table. Args: {\'table_name\': \'vw_results\', \'reasoning\': "The \'vw_results\' table is likely to contain detailed election results, including votes by party. I need to understand its structure to confirm if it has columns for party and vot

In [10]:
print(response["sanitized_query"])

SELECT
    PARTY_NAME,
    SUM(NUM_VOTES) AS TOTAL_VOTES
FROM
    vw_results
GROUP BY
    PARTY_NAME
ORDER BY
    TOTAL_VOTES DESC
LIMIT 20


In [11]:
for out in agent.sql_expert.execute_read_query.invoke({
    "sql_query": response["sanitized_query"],
    "reasoning": "checking query"
}):
    pass

results = out["content"]
out_message = out["message"]

out_message

2026-03-27 15:17:35,909 [INFO] LocalLLMStack: LLM Reasoning (execute_read_query): checking query


'OK'

In [12]:
results

,PARTY_NAME,TOTAL_VOTES
0,RHDP,1058772.0
1,INDEPENDANT,644703.0
2,PDCI-RDA,182713.0
3,FPI,12998.0
4,ADCI,8891.0
5,MGC,3707.0
6,EDS,3387.0
7,CODE,3291.0
8,UNPR,2571.0
9,AIDE,1440.0


In [13]:
intent = agent._get_intent(user_prompt=query)
print(intent)

out = agent._interpret_results(
    user_prompt=query,
    data=results,
    intent=intent
)

pprint(out) 

2026-03-27 15:17:36,023 [INFO] LocalLLMStack: Formatting input to LLM
QueryIntent.AGGREGATION
2026-03-27 15:17:39,871 [INFO] LocalLLMStack: Interpreting results...
2026-03-27 15:17:39,882 [INFO] LocalLLMStack: Prompting LLM
('The total votes by party show that RHDP received the highest number of votes '
 'with over 1.05 million, significantly leading the others. INDEPENDANT came '
 'in second with 644,703 votes. The remaining parties account for a much '
 'smaller share, with the majority of votes concentrated in the top two '
 'parties. The distribution indicates a highly polarized electorate, with a '
 'clear dominance by RHDP and INDEPENDANT.')


In [26]:
query = "Which region has the most voters?"
intent = QueryIntent.RANKING  # Manually set for the test
final_sql_query = None

if CFG.IS_STREAM:
    print(f"--- Starting Stream for: {query} ---\n")
    
    for chunk in agent.sql_expert.generate_sql(query, intent):
        
        if chunk["type"] == "token":
            print(chunk["content"], end="", flush=True)
            
        elif chunk["type"] == "action":
            print(f"\n\n🛠️  [ACTION]: {chunk['content']}")
            
        elif chunk["type"] == "final_sql":
            print(f"\n\n🚀 [FINAL SQL]:\n{chunk['content']}")
            final_sql = chunk['content']
    
    print("\n--- Stream Finished ---")
else:
    for out in agent.sql_expert.generate_sql(query, intent):
        print(out)
        response = out

2026-03-12 02:29:52,272 [INFO] LocalLLMStack: Attempting SQL generation...(Max iterations=18)
{'type': 'status', 'content': 'Attempting SQL generation...(Max iterations=18)'}
2026-03-12 02:29:52,272 [INFO] LocalLLMStack: 
Starting iteration 1/18
{'type': 'status', 'content': '\nStarting iteration 1/18'}
2026-03-12 02:30:44,088 [INFO] LocalLLMStack: Extracting tool calls
2026-03-12 02:30:44,091 [INFO] LocalLLMStack: Executing tool: list_tables with args {'reasoning': 'To identify the relevant tables that might contain information about regions and voter turnout, I first need to list all available tables in the database. This will help me determine which table(s) are likely to hold data on regions and voter numbers.'}
{'type': 'status', 'content': "Executing tool: list_tables with args {'reasoning': 'To identify the relevant tables that might contain information about regions and voter turnout, I first need to list all available tables in the database. This will help me determine which t

In [27]:
response

{'type': 'final_sql',
 'content': 'SELECT \n    REGION_NAME,\n    NUM_VOTERS\nFROM \n    vw_turnout\nORDER BY \n    NUM_VOTERS DESC\nLIMIT 50;',
 'sanitized_query': 'SELECT\n    REGION_NAME,\n    NUM_VOTERS\nFROM\n    vw_turnout\nORDER BY\n    NUM_VOTERS DESC\nLIMIT 50'}

In [28]:
response["sanitized_query"]

'SELECT\n    REGION_NAME,\n    NUM_VOTERS\nFROM\n    vw_turnout\nORDER BY\n    NUM_VOTERS DESC\nLIMIT 50'

In [29]:
results, out_msg = agent.sql_expert.execute_read_query.invoke({
    "sql_query": response["sanitized_query"],
    "reasoning": "checking query"
})

print(f"{results.shape=}")
print(f"{out_msg=}")

2026-03-12 02:32:36,822 [INFO] LocalLLMStack: LLM Reasoning (execute_read_query): checking query
results.shape=(50, 2)
out_msg='OK'


In [30]:
results

,REGION_NAME,NUM_VOTERS
0,PORO,142250
1,GBEKE,114172
2,DISTRICTAUTONOMED'ABIDJAN,107134
3,GUEMON,73989
4,DISTRICTAUTONOMED'ABIDJAN,73989
5,TCHOLOGO,40576
6,BAGOUE,36846
7,DISTRICTAUTONOMEDEYAMOUSSOUKRO,36304
8,DISTRICTAUTONOMED'ABIDJAN,33793
9,DISTRICTAUTONOMED'ABIDJAN,33090


In [31]:
intent = agent._get_intent(user_prompt=query)
print(intent)

out = agent._interpret_results(
    user_prompt=query,
    df=results,
    intent=intent
)

pprint(out) 

2026-03-12 02:32:36,975 [INFO] LocalLLMStack: Formatting input to LLM
QueryIntent.RANKING
2026-03-12 02:32:40,103 [INFO] LocalLLMStack: Interpreting results...
2026-03-12 02:32:40,106 [INFO] LocalLLMStack: Prompting LLM
('The region with the most voters is **PORO**, with a total of **142,250** '
 'voters. This significantly outpaces the second-highest region, **GBEKE** '
 '(114,172 voters), highlighting a substantial gap in voter numbers. \n'
 '\n'
 "Notably, **DISTRICTAUTONOMED'ABIDJAN** appears multiple times across the "
 'dataset, with several entries in the 30,000–40,000 voter range, indicating '
 'it is a major electoral area but still behind PORO. Other regions like '
 'GUEMON and TCHOLOGO show repeated entries with lower voter counts, '
 'suggesting possible data duplication or reporting inconsistencies.\n'
 '\n'
 'The data reveals a clear concentration of voters in PORO and GBEKE, with '
 'significant disparities among the rest of the regions—many of which have '
 'fewer than 

In [ ]:
query = "What is the turnout in the region with the most voters?"
intent = QueryIntent.AGGREGATION 

if CFG.IS_STREAM:

    print("--- [STARTING STREAM] ---")
    for update in agent.sql_expert.generate_sql(query, intent):
        
        if update["type"] == "token":
            print(update["content"], end="", flush=True)
            
        elif update["type"] == "action":
            print(f"\n\n🛠️  [TOOL CALL]: {update['content']}")
            
        elif update["type"] == "final_sql":
            print(f"\n\n🚀 [FINAL SQL]:\n{update['content']}")
    
    print("\n--- [STREAM FINISHED] ---")
else:
    response = agent.sql_expert.generate_sql(query, intent)

### Full End-to-End Analytical Run

In [ ]:
vague_query = "Who are the candidates who won the elections in Abidjan?" 
# Logic: Abidjan is a wide region so agent might need clarification

agent._get_intent(user_prompt=vague_query)

2026-03-11 13:06:46,082 [INFO] LocalLLMStack: Formatting input to LLM


<QueryIntent.RANKING: 'ranking'>

In [16]:
agent.rag_expert.search_database.args

{'query': {'title': 'Query', 'type': 'string'}}

In [17]:
results = agent.rag_expert.search_database.invoke({    
    "query": vague_query
    })

2026-03-11 13:06:52,091 [INFO] sentence_transformers.SentenceTransformer: Use pytorch device_name: mps
2026-03-11 13:06:52,091 [INFO] sentence_transformers.SentenceTransformer: Load pretrained SentenceTransformer: google/embeddinggemma-300m


Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

2026-03-11 13:07:03,042 [INFO] sentence_transformers.SentenceTransformer: 14 prompts are loaded, with the keys: ['query', 'document', 'BitextMining', 'Clustering', 'Classification', 'InstructionRetrieval', 'MultilabelClassification', 'PairClassification', 'Reranking', 'Retrieval', 'Retrieval-query', 'Retrieval-document', 'STS', 'Summarization']


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [18]:
for text, chunk_id, score in results:
    print(f"chunk_id {chunk_id}: \t[{score:.2f}] {text}")

chunk_id 173: 	[0.02] In constituency 038 (ABOBO, COMMUNE) of region DISTRICTAUTONOMED'ABIDJAN, ABOBO RENAISSANCE (INDEPENDANT) received 916 votes (0.87%) but has lost.
chunk_id 174: 	[0.01] In constituency 038 (ABOBO, COMMUNE) of region DISTRICTAUTONOMED'ABIDJAN, AGIR AUJOURDHUI POUR BATIR DEMAIN (AIDE) received 1374 votes (1.31%) but has lost.
chunk_id 1125: 	[0.01] In constituency 038 (ABOBO, COMMUNE) of region DISTRICTAUTONOMED'ABIDJAN, UNE CÔTE DIVOIRE EN PAIX, PROSPERE ET SOLIDAIRE (RHDP) received 92947 votes (88.45%) but has lost.
chunk_id 178: 	[0.01] In constituency 038 (ABOBO, COMMUNE) of region DISTRICTAUTONOMED'ABIDJAN, TOUS ENSEMBLE POUR LA CÔTE D'IVOIRE (PDCI-RDA) received 7359 votes (7.0%) but has lost.
chunk_id 1722: 	[0.01] The voter turnout in constituency 048 (DISTRICTAUTONOMED'ABIDJAN) was 21.22%. There were 107529 registered voters and 22226 expressed votes including 235 blank votes (1.06%).
chunk_id 175: 	[0.01] In constituency 038 (ABOBO, COMMUNE) of region DISTR

### get_answer()

In [19]:
print(f"--- Testing Disambiguation for: {vague_query} ---")
try:
    for response in agent.get_answer(vague_query):

        if response.get("type") == "clarification":
            print("Success: Agent identified ambiguity.")
            print("Agent Question:", response["content"])
            print("Options found in DB:", response["options"])

except Exception as e:
    logger.error(f"Failure: Agent failed to process query. {e}", exc_info=True)


--- Testing Disambiguation for: Who the candidates who won the elections in Abidjan? ---
2026-03-11 13:07:17,510 [INFO] LocalLLMStack: Formatting input to LLM
2026-03-11 13:07:21,619 [INFO] LocalLLMStack: Identifying decision route using LLM routing...
2026-03-11 13:07:25,936 [INFO] LocalLLMStack: ✅ Identified route: SQL
2026-03-11 13:07:25,937 [INFO] LocalLLMStack: Generating SQL query
2026-03-11 13:07:25,937 [INFO] LocalLLMStack: Attempting restricted SQL generation...(Max iterations=18)
2026-03-11 13:07:25,938 [ERROR] LocalLLMStack: Query generation error: 'StructuredTool' object is not callable


In [13]:
response

{'type': 'status',
 'content': "Could not retrieve response. 'str' object has no attribute 'hybrid_search'"}

#### Test various queries

In [ ]:
queries = [
    "How many seats did RHDP win?",
    "Who won the elections in tiapum",
    "Top 10 candidates by score in region Nawa.",
    "Participation rate by region",
    "Histogram of winners by party.",
    "distribution of winners per party",
    "Which party did win the most seats?",
    "Show me the distribution of voters per party and per region"
]

for query in queries:
    print(f'>> Query: {query}')
    out = agent.get_answer(user_prompt=query)
    print(f'>> Response: {out["content"]}\n')
    break
